# Module 4 - Week 5 Module Demo: ML Concepts & Workflow
## Live Demo Notebook

**Scenario:**
You've been hired as a data scientist at a Nigerian fintech startup.
They want to predict whether a loan applicant will default - so they can approve or reject applications automatically.

This notebook runs through the complete Week 5 curriculum in one continuous session:

| Section | Topic covered |
|---|---|
| 1. What is this problem? | Topic 1 - What is ML; when to use it vs rules |
| 2. What type of ML problem is this? | Topic 2 - Types of ML (supervised classification) |
| 3. Follow the workflow | Topic 3 - The ML Workflow, step by step |
| 4. Diagnose the first model | Topic 4 - Overfitting & underfitting |
| 5. Build it properly with scikit-learn | Topic 5 - scikit-learn API, Pipeline |
| 6. Final model + interpretation | Topic 6 - Full build, feature importance, predictions |

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded.')

---
# Section 1 - What Is This Problem?
## Topic 1: What is Machine Learning - Rules vs ML

The fintech company currently uses a manual rule:
> *Approve if credit_score ≥ 650 AND income ≥ 200,000 AND employment_years ≥ 2*

Let's first check how well that rule works - then see if ML can do better.

In [ ]:
df = pd.read_csv('loan_applications.csv')
print('Shape:', df.shape)
print(f'Default rate: {df["defaulted"].mean():.1%}')
df.head()

In [ ]:
# The current rule-based system
def rule_approve(row):
    return int(row['credit_score'] >= 650 and
               row['income']       >= 200000 and
               row['employment_years'] >= 2)

df['rule_decision'] = df.apply(rule_approve, axis=1)

# How accurate is the rule at predicting NON-default?
# (we want to approve people who will repay)
rule_correct = ((df['rule_decision'] == 1) & (df['defaulted'] == 0)).sum()
rule_wrong   = ((df['rule_decision'] == 1) & (df['defaulted'] == 1)).sum()

print('Rule-based system on 300 applicants:')
print(f'  Approved and repaid (correct):  {rule_correct}')
print(f'  Approved but defaulted (wrong): {rule_wrong}')
print(f'  Precision among approved: {rule_correct/(rule_correct+rule_wrong)*100:.0f}%')
print()
print('The rule is a starting point - but does it capture all safe applicants?')
print('Can ML do better with all 6 features together?')

---
# Section 2 - What Type of ML Problem Is This?
## Topic 2: Types of Machine Learning

**Questions to answer:**
1. Do we have labels? → YES (`defaulted` column) → **Supervised Learning**
2. What kind of label? → 0 or 1 (binary) → **Classification**
3. How many classes? → 2 → **Binary Classification**

In [ ]:
print('ML Type Identification:')
print()
print('1. Do we have labels?')
print(f'   YES - "defaulted" column exists with {df["defaulted"].nunique()} unique values: {sorted(df["defaulted"].unique())}')
print()
print('2. What kind of label?')
print('   Binary (0 or 1) → Classification, not Regression')
print()
print('3. ML type: SUPERVISED BINARY CLASSIFICATION')
print()
print('Class distribution:')
print(df['defaulted'].value_counts())
print(f'Default rate: {df["defaulted"].mean():.1%} - relatively rare class (15%)')

---
# Section 3 - Follow the ML Workflow
## Topic 3: The ML Workflow

In [ ]:
# WORKFLOW STEP 2: Prepare data - EDA
print('=== Step 2: Data Preparation ===')
print()
print('Missing values:', df.isnull().sum().sum())
print()
print('Feature statistics:')
print(df.drop('defaulted', axis=1).describe().round(0))

In [ ]:
# Feature engineering - create a debt-to-income ratio
df['debt_to_income'] = df['loan_amount'] / (df['income'] * 12)

FEATURES = ['age', 'income', 'loan_amount', 'credit_score',
            'employment_years', 'num_dependants', 'debt_to_income']
TARGET = 'defaulted'

X = df[FEATURES]
y = df[TARGET]

print('Features selected (with engineered debt_to_income):')
for f in FEATURES:
    print(f'  {f}')

In [ ]:
# WORKFLOW STEP 3: Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training: {len(X_train)} | Test: {len(X_test)}')
print(f'Train default rate: {y_train.mean():.1%} | Test default rate: {y_test.mean():.1%}')
print('Stratified split: class proportions preserved.')

In [ ]:
# WORKFLOW STEP 4: Train first model
first_model = DecisionTreeClassifier(max_depth=5, random_state=42)
first_model.fit(X_train, y_train)

# WORKFLOW STEP 5: Evaluate
train_acc = first_model.score(X_train, y_train)
test_acc  = first_model.score(X_test,  y_test)

print(f'First model (Decision Tree, depth=5):')
print(f'  Training accuracy: {train_acc:.2%}')
print(f'  Test accuracy:     {test_acc:.2%}')
print(f'  Gap:               {(train_acc - test_acc):.2%}')

---
# Section 4 - Diagnose the First Model
## Topic 4: Overfitting & Underfitting

In [ ]:
# Build a learning curve to find the best depth
depths = list(range(1, 21))
train_scores, test_scores = [], []

for depth in depths:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    train_scores.append(m.score(X_train, y_train))
    test_scores.append(m.score(X_test, y_test))

best_depth = depths[test_scores.index(max(test_scores))]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_scores, label='Training accuracy', color='steelblue', marker='o', markersize=4)
ax.plot(depths, test_scores,  label='Test accuracy',     color='coral',     marker='o', markersize=4)
ax.axvline(best_depth, color='green', linestyle='--',
           label=f'Best depth = {best_depth}')
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Learning Curve - Loan Default Classifier', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Best max_depth: {best_depth} (test accuracy: {max(test_scores):.2%})')

---
# Section 5 - Build It Properly With scikit-learn
## Topic 5: scikit-learn API - Pipeline

In [ ]:
# Build a proper Pipeline - scaler + model
# Note: Decision Trees don't NEED scaling, but we include it to demonstrate Pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestClassifier(n_estimators=100, max_depth=best_depth,
                                       random_state=42, n_jobs=-1)),
])

pipe.fit(X_train, y_train)

pipe_train_acc = pipe.score(X_train, y_train)
pipe_test_acc  = pipe.score(X_test, y_test)

print('Pipeline (StandardScaler + RandomForest):')
print(f'  Training accuracy: {pipe_train_acc:.2%}')
print(f'  Test accuracy:     {pipe_test_acc:.2%}')
print(f'  Gap:               {(pipe_train_acc - pipe_test_acc):.2%}')
print()
print('Improvement over first Decision Tree model?')
print(f'  First model test accuracy: {test_acc:.2%}')
print(f'  Random Forest test accuracy: {pipe_test_acc:.2%}')

---
# Section 6 - Final Model, Feature Importance, and Predictions
## Topic 6: Building Your First Model

In [ ]:
# Detailed evaluation
y_pred = pipe.predict(X_test)

print('=== Final Model Evaluation ===')
print(classification_report(y_test, y_pred, target_names=['Repaid', 'Defaulted']))

In [ ]:
# Feature importance from the Random Forest inside the Pipeline
rf_model   = pipe.named_steps['model']
importance = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
importance.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Importance - Loan Default Prediction',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 3 most important features:')
for feat, imp in importance.sort_values(ascending=False).head(3).items():
    print(f'  {feat:<20} {imp:.3f}')

In [ ]:
# Predict on 4 new applicants
new_applicants = pd.DataFrame({
    'age':              [28,    45,    35,    52   ],
    'income':           [180000, 650000, 320000, 95000],
    'loan_amount':      [500000, 2000000, 800000, 1500000],
    'credit_score':     [580,   780,    640,   420   ],
    'employment_years': [1,     12,     5,     3     ],
    'num_dependants':   [0,     2,      1,     5     ],
})
new_applicants['debt_to_income'] = new_applicants['loan_amount'] / (new_applicants['income'] * 12)

predictions  = pipe.predict(new_applicants[FEATURES])
probabilities = pipe.predict_proba(new_applicants[FEATURES])[:, 1]  # prob of defaulting

print('=== Loan Application Decisions ===')
print()
names = ['Amara', 'Emeka', 'Funmilayo', 'Ibrahim']
for i, name in enumerate(names):
    decision    = 'REJECT' if predictions[i] else 'APPROVE'
    default_prob = probabilities[i]
    row = new_applicants.iloc[i]
    print(f'{name}:')
    print(f'  Income: NGN {row.income:,} | Loan: NGN {row.loan_amount:,} | Credit: {row.credit_score}')
    print(f'  ML Decision: {decision} | Default probability: {default_prob:.1%}')
    print()

In [ ]:
# Final diagnostic check
print('=== Final Fit Diagnostic ===')
print(f'Training accuracy: {pipe_train_acc:.2%}')
print(f'Test accuracy:     {pipe_test_acc:.2%}')
print(f'Gap:               {(pipe_train_acc - pipe_test_acc):.2%}')
print()
if pipe_train_acc - pipe_test_acc > 0.1:
    print('OVERFITTING - reduce max_depth or collect more data.')
elif pipe_test_acc < 0.75:
    print('UNDERFITTING - add features or try a more powerful algorithm.')
else:
    print('GOOD FIT - model is ready for stakeholder review.')

---
## Week 5 Complete - Full Story Summary

| Topic | What we demonstrated |
|---|---|
| 1 - What is ML | Rules vs ML - ML captures nuance the rules miss |
| 2 - Types of ML | Supervised binary classification - identified from the label column |
| 3 - The Workflow | All 5 steps applied sequentially on real loan data |
| 4 - Overfitting | Learning curve revealed the best depth; diagnosed first model |
| 5 - scikit-learn | Pipeline built correctly - no leakage, clean API |
| 6 - First Model | Random Forest trained, feature importance extracted, predictions made |

**The business outcome:** The fintech startup now has a model that can:
- Automatically score new loan applicants in milliseconds
- Provide a default probability (not just approve/reject)
- Tell the business which factors drive default risk most

**Next - Week 6:** Supervised Learning in depth - Linear Regression, Logistic Regression, Decision Trees, Random Forests, and proper evaluation metrics.